### RAG Pipelines - Data Ingestion to Vector DB Pipeline

In [1]:
import os
from pathlib import Path
from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

C:\Users\Legion\AppData\Local\Temp\ipykernel_28492\3110145442.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader, PyMuPDFLoader
d:\ALL ABOUT CODE\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### Read the pdf inside the directory

project_root = Path.cwd()
pdf_path = Path("D:\ALL ABOUT CODE\RAG\data\pdf\Attention.pdf")

if not pdf_path.exists():
    pdf_path = (project_root / "notebook").parent / "data" / "pdf" / "Assignment.pdf"

loader = PyMuPDFLoader(str(pdf_path))
document = loader.load()

print(f"Loaded {len(document)} document(s) from {pdf_path}")


Loaded 15 document(s) from D:\ALL ABOUT CODE\RAG\data\pdf\Attention.pdf


In [3]:
### Text Splitting get into chunks

def split_documents(document, chunk_size=1000, chunk_overlap=200):
    '''Split document into smaller chunks for better RAG performance'''
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(document)
    print(f"Split {len(document)} document into {len(split_docs)} chunks")
    
    #Show exapmle of chunk
    if split_docs:
        print(f"\n Example chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
        
    return split_docs

In [5]:
chunks = split_documents(document)

Split 15 document into 52 chunks

 Example chunk:
Content: Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
...
Metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'source': 'D:\\ALL ABOUT CODE\\RAG\\data\\pdf\\Attention.pdf', 'file_path': 'D:\\ALL ABOUT CODE\\RAG\\data\\pdf\\Attention.pdf', 'total_pages': 15, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'trapped': '', 'modDate': 'D:20240410211143Z', 'creationDate': 'D:20240410211143Z', 'page': 0}


Embedding and VectorStoreDB

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name (str, optional): _description_. Defaults to "all-MiniLM-L6-v2".
        """
        
        self.model_name = model_name
        self.model = None
        self._load_model()
        
    def _load_model(self):
        """Load the SentenceTransformer model."""
        try:
                print(f"Loading embedding model: {self.model_name}")
                self.model = SentenceTransformer(self.model_name)
                print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
                print(f"Error loading embedding model: {e}")
                raise
    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts.
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dimension)
        """
        if not self.model:
            raise ValueError("Model is not loaded.")

        print(f"Generatiog embeddings for {len(texts)} texts....")
        embeddings = self.model.encode(texts, show_progress_bar = True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    
## Initialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4429.54it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\Legion\AppData\Local\Temp\ipykernel_28492\1751863325.py:21: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### Vector Store

In [8]:
class VectorStore:
    """
    Manages document embeddings in a ChromaDB vector store.
    """

    def __init__(
        self,
        collection_name: str = "pdf_document",
        persist_directory: str = "../data/vector_store",
    ):
        """
        Initialize the vector store.

        Args:
            collection_name: Name of the ChromaDB collection.
            persist_directory: Directory to persist the vector store.
        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """
        Initialize ChromaDB client and collection.
        """

        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"},
            )

            print(f"Vector store initialized. " f"Collection: {self.collection_name}")
            print(f"Existing documents in collection: " f"{self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_document(self, document: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store.

        Args:
            document: List of LangChain documents.
            embeddings: Corresponding embeddings for the documents.
        """

        if len(document) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(document)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        document_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(document, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            document_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=document_text,
            )

            print(f"Successfully added {len(document)} " f"documents to vector store.")
            print(f"Total documents in collection: " f"{self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise
        
vectorstore = VectorStore()
vectorstore

Vector store initialized. Collection: pdf_document
Existing documents in collection: 82


In [9]:
chunks

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'source': 'D:\\ALL ABOUT CODE\\RAG\\data\\pdf\\Attention.pdf', 'file_path': 'D:\\ALL ABOUT CODE\\RAG\\data\\pdf\\Attention.pdf', 'total_pages': 15, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'trapped': '', 'modDate': 'D:20240410211143Z', 'creationDate': 'D:20240410211143Z', 'page': 0}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\nai

In [10]:
### Convert the text to embeddings
texts = [doc.page_content for doc in chunks]
texts

### Generate the embeddings
embeddings = embedding_manager.generate_embeddings(texts)
embeddings

## Store in the vector database
vectorstore.add_document(chunks, embeddings)

Generatiog embeddings for 52 texts....


Batches: 100%|██████████| 2/2 [00:02<00:00,  1.24s/it]

Generated embeddings with shape: (52, 384)
Adding 52 documents to vector store...
Successfully added 52 documents to vector store.
Total documents in collection: 134


Retriever Pipeline From VectorStore

In [11]:
class RAGRetriever:
    """Handles query-based retireval from the vector stoer"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """Initialize the retiever

        Args:
            vector_store = Vector store containing document embeddings
            embedding_manager = Manager for generating query embeddings
        """

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(
        self, query: str, top_k: int = 5, score_threshold: float = 0.0
    ) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata

        """
        print(f"Retrieveing doucments for query: '{query}'")
        print(f"top_k: {top_k}; score_threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
            )

            retrieved_docs = []

            if results.get('documents') and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(
                    zip(ids, documents, metadatas, distances)
                ):
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "document": document,
                            "similarity_score": similarity_score,
                            "metadata": metadata,
                            "distance": distance,
                            "rank": i + 1,
                        })

                print(f"Retireved {len(retrieved_docs)} documents")
                return retrieved_docs

            print("No documents retrieved for the query.")
            return []

        except Exception as e:
            print(f"Error retrieving documents: {e}")
            return []

rag_retriever = RAGRetriever(vectorstore, embedding_manager)

In [12]:
rag_retriever

In [13]:
rag_retriever.retrieve("Can you summarize the task?")

Retrieveing doucments for query: 'Can you summarize the task?'
top_k: 5; score_threshold: 0.0
Generatiog embeddings for 1 texts....


Batches: 100%|██████████| 1/1 [00:00<00:00, 47.62it/s]

Generated embeddings with shape: (1, 384)
Retireved 2 documents


[{'id': 'doc_77237b3f_6',
  'content': 'be articulated to you ahead of the summative assessment \n \nBased on the AI generated response, write a 4000-word Summative \nAssessment in the following areas: \n▪ Critical analyse the AI-generated response demonstrating engagement with \nrelevant literature, critical thinking, and application of healthcare management \nprinciples. \n(Please note students are not required to critique the AI–generative tool \ni.e. Microsoft Copilot).  \n \nSuggested Structure for Summative Assignment \nIntroduction:  \n▪ Introduce the concept of resource management and its importance for a \nhealthcare organisation.  \n▪ State the purpose of the essay and the significance of your chosen resource \nmanagement issue in relation to managing in a healthcare organisation.  \n▪ State how you intend to accomplish the assignment task using the Microsoft \nCopilot chat generated response.',
  'document': 'be articulated to you ahead of the summative assessment \n \nBased

Integration Vectordb Context pipeling With LLM Output

In [14]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from dotenv import load_dotenv

load_dotenv()

gemini_api_key = os.getenv("GEMINI_API_KEY")

llm = ChatGoogleGenerativeAI(
    google_api_key=gemini_api_key,
    model="gemini-3.6-flash",
    temperature=0.2,
    max_output_tokens=500
)

In [15]:
def rag_simple(query, retriever, llm, top_k=3):
    ## retriever the context
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc["content"] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."

    ##generate the answer using GROQ LLM
    prompt = """Use the following context to answer the question concisely.
    Context:
    {context}

    Question: {query}

    Answer: """

    response = llm.invoke(prompt.format(context=context, query=query))
    return response.content

answer = rag_simple("What is attention mechanism", rag_retriever, llm)
print(answer)

Retrieveing doucments for query: 'What is attention mechanism'
top_k: 3; score_threshold: 0.0
Generatiog embeddings for 1 texts....


Batches: 100%|██████████| 1/1 [00:00<00:00, 31.01it/s]
d:\ALL ABOUT CODE\RAG\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Generated embeddings with shape: (1, 384)
Retireved 3 documents
[{'type': 'text', 'text': 'Based on the provided context, an attention mechanism (or function) is a mapping of a query and a set of key-value pairs to an output—where the query, keys, values, and output are all vectors—and the output is computed as a weighted sum.', 'extras': {'signature': 'EpoJCpcJAWkUfRP8N2QznCYbbiaF8X3gD4WnyfZbEAzywcMaR98EpZpj+QdBRmmG5WKW6Kg8+XQ6zFl22So6pZbFrwa5bcQrhrAB/OPRnmk2B2SS4ZUkQb3/rlaon1LlJR/Jhy/2zYRVRnKS5qTU7tLqJ0anUsVHImIZ6EqlImTPgXpR/37+gIAdYo/771ChP4ojXmb3K5M1qrLK6ofUUx3Pnlp+BIMujdYe6/kf5v3KP4dze/EYZiF2QfefCkwrdvuaOMb2aaad6rkyblODEMlVu0uLH6ACjED5VyQX4ioNQXE5JqHux+C4BfdOoqgq3G02I232VTxIEou9olJoj8jjJoYHJfl8IecEMA2nh2Q8557FSKXjylWm4Ob+zc1iNXa24cVaw4UjempfaB0CCEFu6Bt3VzTSTbHA5y02wQfTyER7S/BfQoD0Uz4/NPL1Pav5soVwFNFR6EhlAF9ak3MzQKiPQzoUbMkYySfDqdzQ1kt9nKPCO4VXPdQDKQ0FWGxcAYYqs1lqZ1R5Rdz/7vccSxUwkru3e/R0eY96iee8ZCUAwrjQdFLbqvBC7wHBhtGqnJpM59cLdEcF7I/mKbOvLeycdilviC5SO3pu8USqKvCbr4qjjWW4avVmoUOD8Ylh

RAG Enhanced Pipeline

In [22]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context

    """
    
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {
            "answer": "No relevant context found to answer the question.",
            "sources": [],
            "confidence_score": 0.0,
            "context": ""
        }
        
    ### Prepare context and sources
    
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
            'source' : doc['metadata'].get('source_file', doc['metadata'].get('source','unknown')),
            'similarity_score' : doc['similarity_score'],
            'page': doc['metadata'].get('page', 'unknown'),
            'preview': doc['content'][:120] + ''
        }
    for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
        
    #generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
        
    output = {
            'answer' : response.content,
            'sources' : sources,
            'confidence' : confidence
        }
    if return_context:
            output['context'] = context
    return output
    
    #Example usuage
result = rag_advanced("Scaled Dot-Product Attention", rag_retriever, llm, top_k=3, min_score=0.2, return_context=True)
print("Answer", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context:", result['context'])

Retrieveing doucments for query: 'Scaled Dot-Product Attention'
top_k: 3; score_threshold: 0.2
Generatiog embeddings for 1 texts....


Batches: 100%|██████████| 1/1 [00:00<00:00, 67.67it/s]

Generated embeddings with shape: (1, 384)
Retireved 3 documents



d:\ALL ABOUT CODE\RAG\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Answer [{'type': 'text', 'text': '**Scaled Dot-Product Attention** is an attention mechanism that calculates output values by taking queries ($Q', 'extras': {'signature': 'EvYNCvMNAWkUfRPT0eTLGN3AtQFZ+W2UJLKhY+QxL/di5O5qfo0uGKGGkeupLynXtWvrwjL/ECWv7VxYC1b5Ts5vvoDOz2XMrQg1GpGP0O+Bfyd3kxQXFqhtBDFhovwKMZFjYIz/udSQqsZzTWg72r8gOMIEsN8orjD9ZZkGImEYq3ie8f6rskLrtdelybNkWrQSOq0JPJcjRxw7j8aHUI0SWuJrerOb/aYm3ABQQs0ueuWSBPpgiKP8h0SGi2UUZjtMubMYTiv8Tcq4mn25TysDF4vfWL5ADChacw15eHxT5UIiivFLGr/2DDoG83fptoUa3uVv/ydNc8jEWZadhRfZpp71M8NtHedPLQUtX6BAXPkVPlhtluZuc/6U+063WH1mh++N7OGoLkP5nw98/zyjAAjMS2lsmTtkIYAa+X5sGdJlWuTocqCq+TgifxybMnDCvUw47604sI0n/mqtGljC/XlLXS99IKI6TFJZSX4mPm0PNO7r7M3L/wo7+vzWpjAAyxQUteQ/AmjJI2244ZwKu1lHb2ybeF6vNvwOEgP3W3PQvXzlzWz/ohkMXQfmm2aBuh3y1KlzyfHaACDEFxrob90FQphgAt3GlTa5VW30nXGjUOzvpwQlXyOPaOWWOcMVqJ80XqbIduFoEPVu8lncK+EhhOfNqLnT04kAWvXzcFoMAoxjN1HL77xwoU0Am6nqf911qH8I5CgGxJEjWrHPE7ZeOzVrA3lmIl95xIvJl4dOmxrDKEuxy99KruQY5bs/ibNlqrW/mXfxkF5uArUeOhuRlRfic0zSJNTK8KTmD7U8vUg8NvDW5uDj